# 02 — Pipeline de modelagem

Treino reproduzível com a mesma lógica de `src/train.py`: IMC, `ColumnTransformer`,
comparação Random Forest vs. Gradient Boosting e serialização do campeão.

In [ ]:
from pathlib import Path
import sys

import joblib
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "Obesity.csv").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data_pipeline import (
    LABEL_PT,
    build_model_pipeline,
    candidate_estimators,
    load_raw_dataset,
    split_xy,
)

df = load_raw_dataset(ROOT / "data" / "Obesity.csv")
X, y = split_xy(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
print("Atributos:", list(X.columns))


## Comparação dos candidatos

In [ ]:
rows = []
fitted = {}
for name, estimator in candidate_estimators().items():
    pipe = build_model_pipeline(estimator)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, pred)
    fitted[name] = (pipe, pred)
    rows.append({"modelo": name, "acuracia_teste": acc})

cmp = pd.DataFrame(rows).sort_values("acuracia_teste", ascending=False)
display(cmp)
champion_name = cmp.iloc[0]["modelo"]
champion, y_pred = fitted[champion_name]
print("Campeão:", champion_name)


In [ ]:
print(classification_report(y_test, y_pred, target_names=[LABEL_PT[c] for c in champion.classes_]))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=[LABEL_PT[c] for c in champion.classes_], ax=ax, xticks_rotation=45
)
ax.set_title("Matriz de confusão — conjunto de teste")
plt.tight_layout()


## Importância dos atributos (se o campeão for baseado em árvores)

In [ ]:
clf = champion.named_steps["classifier"]
pre = champion.named_steps["preprocessor"]
if hasattr(clf, "feature_importances_"):
    names = pre.get_feature_names_out()
    imp = (
        pd.DataFrame({"feature": names, "importance": clf.feature_importances_})
        .sort_values("importance", ascending=False)
        .head(15)
    )
    display(imp)
    imp.iloc[::-1].plot.barh(x="feature", y="importance", figsize=(8, 5), legend=False)
    plt.title("Top 15 atributos")
    plt.tight_layout()


## Serialização para o aplicativo

In [ ]:
out = ROOT / "app" / "model.joblib"
out.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(champion, out)
print("Modelo salvo em", out)
print("Acurácia de teste do campeão: {:.2%}".format(accuracy_score(y_test, y_pred)))
